In [ ]:
from casadi import *
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.io
import pickle
import lightning as L
from sklearn.preprocessing import MinMaxScaler

# Model definition

In [ ]:
PARAMETERS = [
    [1.327, 0.0034, 44.223, 2.308, 56.001, 21.840],
    [2.011, 0.0063, 54.185, 2.410, 40.004, 14.624],
    [0.757, 0.0010, 25.271, 1.737, 52.202, 21.516],
    [1.252, 0.0027, 58.349, 2.961, 59.502, 25.429]
]

def ruan_model(parameters):
    # Define the Ruan model from (Ruan et al., 2017), (Abuin et al., 2020)

    # Parameters from (Abuin et al., 2020), identified from 10 in-silico adults
    # in order: endogenous glucose production, glucose effectiveness, insulin sensitivity, carbohidrate factor, time-to-maximum effective insulin concentration, time-of-maximum appearance rate of glucose in gut)

    theta_0 = parameters[0]
    theta_1 = parameters[1]
    theta_2 = parameters[2]
    theta_3 = parameters[3]
    theta_4 = parameters[4]
    theta_5 = parameters[5]

    A = DM([
        [ -theta_1  , -theta_2      , 0         , theta_3   , 0],
        [ 0         , -1/theta_4    , 1/theta_4 , 0         , 0],
        [ 0         , 0             , -1/theta_4, 0         , 0],
        [ 0         , 0             , 0         , -1/theta_5, 1/theta_5],
        [ 0         , 0             , 0         , 0         , -1/theta_5],    
        ])

    # input matrix of u(t) i.e. insulin infusion rate (both basal and boluses) [U/min]
    Bu = DM([[0], [0], [1/theta_4], [0], [0]])
    # input matrix of r(t) u.e. carbohydrate intake rate (CHO) [g/min]
    Br = DM([[0], [0], [0], [0], [1/theta_4]])
    # basal steady-state endogenous glucose production
    E = DM([[theta_0], [0], [0], [0], [0]])
    # input (Blood Glucose Levels)
    C = DM([1, 0, 0, 0, 0])

    # stack the two input vectors into a single input matrix since casadi integrators only allow for one control vector
    B = horzcat(Bu, Br)

    # print(A)
    # print(B)
    # print(E)

    # print(C)

    return {
        "A": A,
        "B": B,
        "Bu": Bu,
        "Br": Br,
        "E": E,
        "C": C
    }

# Input preparation

In [ ]:
# utility function to extract
 
def extract_data_from_mat(filename="Data/DMMSR_dataset_OnlyBasal.mat"):
    data = scipy.io.loadmat(filename)
    #print(data["DMMS_export"])

    for i, p in enumerate(data["DMMS_export"][0]):
        # print(np.array(p[1]).T[0])
        
        df = pd.DataFrame({
            "insulin_U" : np.array(p[3]).T[0],
            "meal_g"    : np.array(p[4]).T[0],
            "cgm"       : np.array(p[2]).T[0],
            "time_min"  : np.array(p[1]).T[0]
        })

        name = f"/DMMSR_dataset_OnlyBasal_patient_{i+1}"
        df.to_pickle("Data/pickle"+name+".pkl")
        df.to_csv("Data/CSV"+name+".csv")


In [ ]:
def load_uva_padova(filename="Data/pickle/DMMSR_dataset_OnlyBasal.pkl"):
    with open(filename, 'rb') as pkl:
        df = pickle.load(pkl)

    u_seq = np.array(df["insulin_U"].astype(float))
    r_seq = np.array(df["meal_g"   ].astype(float))
    cgm_uva = np.array(df["cgm"   ].astype(float))
    tout = np.array(df["time_min"   ].astype(int))

    return {
        "t_out": tout,
        "insulin": u_seq,
        "meals": r_seq,
        "cgm": cgm_uva
    }


In [ ]:
def ruan_simulation(parameters, tout, u_insulin, u_meals, plot=False):
    ruan = ruan_model(parameters)
    A = ruan["A"]
    B = ruan["B"]
    E = ruan["E"]

    # continuous time integration with pre-programmed meals and insulin
    # slicer = slice(0,-1)

    x = MX.sym('x', 5)
    U = MX.sym('u', 2)

    ode = {'x':x, 'u': U, 'ode': A@x + B@U + E}
    F = integrator('F','rk',ode, 0, tout ,{})

    # integration only allows for one control vector U, so the u, r have to be stacked
    U_seq = horzcat(u_insulin, u_meals).T

    # integrate
    res = F(x0=[100,0,0,0,0], u=U_seq)

    cgm_ruan = np.array(res['xf'][0,:].T)

    if plot:
        fig, ax = plt.subplots()
        ax.plot(tout, cgm_ruan) # np.array(df["cgm"])[slicer]

    return cgm_ruan

# Dataset preparation

For one desired patient, load the corresponding UVA/Padova data and perform a simulation using the Ruan model, the collect the needed data for the dataset

In [ ]:
# by default the dataset is downsampled to every 5minutes
# since by downsampling information on the meals could be lost, move the meal to the closest multiple of downsample_min
def compute_dataset(patient_id = 1, downsample_min=5, stride=0, plot=False):
    if stride >= downsample_min: stride = downsample_min - 1

    uvap_data = load_uva_padova(filename=f"Data/pickle/DMMSR_dataset_OnlyBasal_patient_{patient_id}.pkl")
    cgm_ruan = ruan_simulation(PARAMETERS[patient_id], uvap_data["t_out"], uvap_data["insulin"], uvap_data["meals"], plot=plot)

    # numpy array, phase shift
    def sample_with_phase(array, downsample_min, stride):
        if stride > 0:
            V = []
            for i in range(0, downsample_min, stride):
                V.append(array[i::downsample_min])
            return np.concatenate(V)
        else: return array[::downsample_min]

    # and cast to pytorch tensors
    time_downsample = torch.tensor(sample_with_phase(uvap_data["t_out"], downsample_min, stride), dtype=torch.float32).reshape(1, -1)
    insulin_t = torch.tensor(sample_with_phase(uvap_data["insulin"], downsample_min, stride), dtype=torch.float32).reshape(1, -1)
    cgm_uva_t = torch.tensor(sample_with_phase(uvap_data["cgm"], downsample_min, stride), dtype=torch.float32).reshape(1, -1)
    cgm_ruan_t = torch.tensor(sample_with_phase(cgm_ruan, downsample_min, stride), dtype=torch.float32).reshape(1, -1)

    # due to downsampling, we might miss a meal in the dataset, but see the effects in CGM
    # copy a meal in the -meal_clone_window, +meal_clone_window interval around its index to be sure it gets sampled
    meal_clone_window = downsample_min // 2
    dim = len(uvap_data["meals"])
    # print(dim)

    shifter = np.eye(dim)
    for i in range(1, meal_clone_window):
        shifter += np.diag(np.ones(dim-i), k=i)
        shifter += np.diag(np.ones(dim-i), k=-i)

    meals_shifted = shifter @ uvap_data["meals"]

    # print(shifter.shape, r_seq.shape, meals_shifted.shape)
    meals_t = torch.tensor(sample_with_phase(meals_shifted, downsample_min, stride), dtype=torch.float32).reshape(1, -1)

    return {
        "time": time_downsample,
        "insulin": insulin_t,
        "meals": meals_t,
        "cgm_ruan": cgm_ruan_t,
        "cgm_uva": cgm_uva_t
    }

def concat_datasets(datasets):
    
    return {
        "time": torch.cat([x["time"] for x in datasets], dim=1),
        "insulin": torch.cat([x["insulin"] for x in datasets], dim=1),
        "meals": torch.cat([x["meals"] for x in datasets], dim=1),
        "cgm_ruan": torch.cat([x["cgm_ruan"] for x in datasets], dim=1),
        "cgm_uva": torch.cat([x["cgm_uva"] for x in datasets], dim=1),
    }

LSTM layers require sequences as inputs.
Divide the data into input (insulin by controller, meals) and output sequences. Use some days for training, some for validation, some for testing.

In [ ]:
import torch
from torch import nn
from torch.nn.utils.rnn import pack_sequence
from torch.utils.data import Dataset, DataLoader

# device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

torch.set_default_dtype(torch.float32)
torch.set_default_device("cuda")
torch.get_default_device()
torch.set_float32_matmul_precision('high')

batch_size = 64
seq_len = 70

# Two inputs: u,r
# Create packed sequences of multiple u,r
# Each sequence is obtained by sampling the dataset for 3hours. No overlap between sequences
# https://docs.pytorch.org/docs/2.12/generated/torch.nn.utils.rnn.pack_sequence.html#torch.nn.utils.rnn.pack_sequence
# https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html

class SequenceDataset(Dataset):
    def __init__(self, insulin, meals, cgm_ruan, cgm_uva, seq_length=50):
        assert max(insulin.shape) == max(meals.shape) == max(cgm_ruan.shape) == max(cgm_uva.shape)
        
        # add a current state reference to the data
        # in the MPC formulation, one should have the real cgm levels measured from the patient
        # or TODO: does it make more sense to use the levels computed by ruan?
        # TODO: does it make sense to pass the entire state of the ruan model? Could we have access to it in MPC?

        # prenormalize data
        # TODO: when considering boluses insulin will also need to be normalized
        # since now we're only considering basal levels, normalization would return nan
        meals_mean = meals.mean(dim=1, keepdim=True)
        meals_std = meals.std(dim=1, keepdim=True)
        self.cgm_uva_mean = cgm_uva.mean(dim=1, keepdim=True)
        self.cgm_uva_std = cgm_uva.std(dim=1, keepdim=True)

        print(self.cgm_uva_mean.shape, self.cgm_uva_mean)
        print(self.cgm_uva_std.shape, self.cgm_uva_std)

        # data = torch.cat((insulin, meals, cgm_uva), dim=0)
        data = torch.cat((insulin, meals, (cgm_uva-self.cgm_uva_mean)/self.cgm_uva_std), dim=0)

        self.len = max(data.shape)-seq_length-1
        
        # split into sequencies of length seq_length and with a stride of 1
        diff = cgm_uva - cgm_ruan
        # print(diff.shape, diff)
        self.y = diff[:,:self.len].T
        
        x = []
        for i in range(self.len):
            _x = data[:,i:(i+seq_length)].T
            x.append(_x)
        
        self.x = torch.stack(x, dim=0)

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

def get_dataset_loader(patient_id=1, downsample_min=5, train_percent = 0.75, val_percent = 0.15,  test_percent = 0.1):
    get_dataset_loader(compute_dataset(patient_id, downsample_min), train_percent = 0.75, val_percent = 0.15,  test_percent = 0.1)

def get_dataset_loader(dataset_load, train_percent = 0.75, val_percent = 0.15,  test_percent = 0.1):
    # Associate each sequence with its respective output
    dataset = SequenceDataset(insulin=dataset_load["insulin"], meals=dataset_load["meals"], cgm_ruan=dataset_load["cgm_ruan"], cgm_uva=dataset_load["cgm_uva"], seq_length=seq_len)

    train_idx = int(len(dataset) * train_percent)
    val_idx = train_idx + int(len(dataset) * val_percent)
    test_idx = val_idx + int(len(dataset) * test_percent)

    train_dataset = torch.utils.data.Subset(dataset, range(train_idx))
    val_dataset = torch.utils.data.Subset(dataset, range(train_idx, val_idx))
    test_dataset = torch.utils.data.Subset(dataset, range(val_idx, test_idx))

    # train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.75, 0.15, 0.10],
    #                                            generator=torch.Generator(device='cuda'))

    # Dataset loader
    train_loader = torch.utils.data.DataLoader(dataset=train_dataset, 
                                            batch_size=batch_size,
                                            shuffle=False)

    val_loader = torch.utils.data.DataLoader(dataset=val_dataset, 
                                            batch_size=batch_size,
                                            shuffle=False)

    test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                            batch_size = 999999,
                                            shuffle=False)

    return {
        "full_dataset": dataset,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "train_sidx": 0,
        "val_sidx": train_idx,
        "test_sidx": val_idx,
        "train_eidx": train_idx-1,
        "val_eidx": val_idx-1,
        "test_eidx": test_idx-1
    }

# NN structure

The NN is made of two LSTM layers and two FC layers

In [ ]:
from lightning.pytorch.callbacks import Callback, EarlyStopping
from lightning.pytorch.callbacks.early_stopping import EarlyStoppingReason

# Lightning Module for LSTM Model
# https://lightning.ai/lightning-ai/templates/time-series-forecasting-with-pytorch-lightning?section=featured
class NN(L.LightningModule):
    def __init__(self, input_dim=2, hidden_dim=150, output_dim=1, lstm_layers=1, dropout=0.2):
        super().__init__()
        self.save_hyperparameters()
        # torch.manual_seed(2)

        # multiple LSTM layers
        # The first layer has two inputs (insulin, meal) and some hidden states
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=lstm_layers, dropout=dropout, batch_first=True)

        # one readout layer
        # the output is the correction to a single sampling time, so dimension 1
        self.fc1 = torch.nn.Linear(hidden_dim, hidden_dim)    
        self.activation = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(hidden_dim,output_dim)

        self.h_0 = None
        self.c_0 = None #tuple initial (hidden, cell)

    def forward(self, x):
        # TODO: this is probably too slow, faster way to avoid this? Edit: does not appear faster without this
        # TODO: this does not even seem to lower the loss, probably the input sequences are already long enough for the network to figure out long term phenomena
        # if self.h_0 is not None and self.c_0 is not None and self.h_0.shape[1] == x.shape[1]:
        #     res, (self.h_0, self.c_0) = self.lstm(x.squeeze(), (self.h_0.detach(), self.c_0.detach()))
        # else:
            # res, (self.h_0, self.c_0) = self.lstm(x.squeeze())
        res, (self.h_0, self.c_0) = self.lstm(x.squeeze())

        res = self.fc1(res[:, -1, :])
        res = self.activation(res)
        res = self.fc2(res)
        return res

        # _, (hidden, _) = self.lstm(x)
        # return self.fc(hidden[-1])

    def training_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs.unsqueeze(-1)).reshape((-1, 1))

        loss = torch.nn.functional.mse_loss(outputs, targets)
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs.unsqueeze(-1)).reshape((-1, 1))

        loss = torch.nn.functional.mse_loss(outputs, targets)
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        inputs, targets = batch
        outputs = self(inputs.unsqueeze(-1)).reshape((-1, 1))

        loss = torch.nn.functional.mse_loss(outputs, targets)
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.0002)

# hidden states size is a hyperparameter. The real dynamics of the Ruan model has five state components
# using PyTorch lightning

model = NN(input_dim=3, lstm_layers=2, dropout=0.2, hidden_dim=200)

In [ ]:

class EpochCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        pl_module.h_0 = None
        pl_module.c_0 = None

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1, stopping_threshold=5, patience=50, verbose=False, mode="min")
epoch_callback = EpochCallback()

# trainer = L.Trainer(max_epochs=2500, callbacks=[epoch_callback, early_stop_callback])
trainer = L.Trainer(max_epochs=2500, callbacks=[early_stop_callback], accelerator="gpu")

# loader1 = get_dataset_loader(patient_id=1, downsample_min=5)
# trainer.fit(model, loader1["train_loader"], loader1["val_loader"])

# train_loader = get_dataset_loader(concat_datasets([compute_dataset(patient_id=1, downsample_min=10), compute_dataset(patient_id=2, downsample_min=10)]), train_percent=1.0, val_percent=0.0, test_percent=0.0)
# val_loader = get_dataset_loader(compute_dataset(patient_id=3, downsample_min=10), train_percent=0.0, val_percent=1.0, test_percent=0.0)

dataset = compute_dataset(patient_id=1, downsample_min=5, stride=1, plot=True)
loader = get_dataset_loader(dataset, train_percent=0.178, val_percent=1/28, test_percent=1/28)

# trainer.fit(model, loader["train_loader"], loader["val_loader"])

model = NN.load_from_checkpoint("lightning_logs/version_153/checkpoints/epoch=292-step=32816.ckpt")
model.eval()

# Access human-readable message
if early_stop_callback.stopping_reason_message:
    print(f"Details: {early_stop_callback.stopping_reason_message}")

In [ ]:
# TODO: replace all this with: trainer.test(dataloaders=test_dataloaders)
x,y = next(iter(loader["test_loader"]))
x = x[:288,:,:]
y = y[:288,:]
ym = model(x)

print(x.shape, y.shape, ym.shape)

y_ = y.squeeze().detach().cpu().numpy().reshape((-1, 1))
ym_ = ym.squeeze().detach().cpu().numpy().reshape((-1, 1))

# The network predicts the correct difference between ruan and UVA/Padova with very small error
fig, ax = plt.subplots()
ax.plot(ym_)
ax.plot(y_)
ax.legend(["y_ref", "y_model"])


In [ ]:
test_meal = x[:,0,1].cpu().numpy()
test_insulin =  x[:,0,0].cpu().numpy()
test_time = np.array([i for i in range(x.shape[0])])

print(test_time.shape, test_insulin.shape, test_meal.shape)
ruan = ruan_simulation(PARAMETERS[0], test_time , test_insulin, test_meal).reshape((-1, 1))
std =  loader["full_dataset"].cgm_uva_std.cpu().numpy()
mean = loader["full_dataset"].cgm_uva_mean.cpu().numpy()
uva = x[:,0,2].cpu().numpy().reshape((-1, 1))
uva_renorm = uva*std + mean

print(ruan.shape, uva.shape, std.shape, mean.shape, ym_.shape)

# ruan = dataset["cgm_ruan"].squeeze()[-l:].cpu().numpy()
# uva = dataset["cgm_uva"].squeeze()[-l:].cpu().numpy()

corrected = ym_+ruan

print(corrected.shape)

# However there is some mismatch in time between the neural-corrected output and the real one
# but it could be just that we are taking the wrong indices (phase of -seq_len)
fig, ax = plt.subplots()
ax.plot(corrected)
ax.plot(uva_renorm)
ax.plot(ruan)
ax.legend(["corrected", "uva", "ruan"])